In [61]:
import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib

import sklearn


from copy import deepcopy
random_state: int = 42

pd.set_option('display.max_columns', None)

## import local lib

import api.utils



In [26]:
PROJECT_ROOT = pathlib.Path.cwd().parent
data_path = PROJECT_ROOT / 'data' / 'processed' / 'processed_data.csv'

In [28]:
_processed_df = pd.read_csv( data_path )
_processed_df.shape

(8849, 12)

In [29]:
processed_df = deepcopy( _processed_df )
processed_df = processed_df.drop( columns= ['start_time', 'end_time', 'modified_date'], errors= 'ignore'  )

processed_df.head(2)


,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,source_type,state,area,pop2020
0,214000.0,0.513,138.0,0.345,417000.0,gas,Alabama,12.899239,5024279.0
1,204000.0,0.513,138.0,0.329,398000.0,gas,Alabama,12.899239,5024279.0


In [30]:
## setting proper dtypes
num_cols = [
    'emissions_quantity', 'capacity', 'capacity_factor', 'activity',
    'area', 'pop2020'
]

cat_cols = ['state', 'source_type']

for c in num_cols:
    if c in processed_df.columns:
        processed_df[c] = pd.to_numeric(processed_df[c], errors='coerce')

for c in cat_cols:
    if c in processed_df.columns:
        processed_df[c] = processed_df[c].astype('string').str.strip()


In [31]:
processed_df[ num_cols ].describe()
processed_df.head()

,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,source_type,state,area,pop2020
0,214000.0,0.513,138.0,0.345,417000.0,gas,Alabama,12.899239,5024279.0
1,204000.0,0.513,138.0,0.329,398000.0,gas,Alabama,12.899239,5024279.0
2,209000.0,0.515,138.0,0.335,406000.0,gas,Alabama,12.899239,5024279.0
3,201000.0,0.513,138.0,0.324,392000.0,gas,Alabama,12.899239,5024279.0
4,113000.0,0.452,87.0,0.328,250000.0,gas,Texas,65.363350,29145505.0


In [38]:
'  asd  a aa'.replace( ' ', '' )
processed_df['state'].unique()
processed_df['source_type'].unique()


<StringArray>
['gas', 'oil', 'coal', 'other_fossil', 'biomass', 'waste']
Length: 6, dtype: string

In [74]:

feature_df = (  deepcopy( processed_df )
	# 1) transforms + ratios 
	.assign(
		log1p_activity=         lambda df:  np.log1p(  df['activity']  ),
		log1p_capacity=         lambda df:  np.log1p(  df['capacity']  ),
		log1p_pop2020=          lambda df:  np.log1p(  df['pop2020']  ),
		log1p_area=             lambda df:  np.log1p(  df['area']  ),
		log1Pop_density=      		lambda df:  np.log1p(  df['pop2020'] / df['area'].replace(  0, np.nan  )  ),
		activity_per_capita=    lambda df:  df['activity'] / df['pop2020'].replace(  0, np.nan  ),
		activity_per_area=      lambda df:  df['activity'] / df['area'].replace(  0, np.nan  ),
		capacity_per_capita=    lambda df:  df['capacity'] / df['pop2020'].replace(  0, np.nan  ),
		capacity_density=       lambda df:  df['capacity'] / df['area'].replace(  0, np.nan  ),

	# 2) power-system structure features 
		potential_output=               lambda df:  df['capacity'] * df['capacity_factor'],
		utilization_ratio=              lambda df:  df['activity'] / (  (df['capacity'] * df['capacity_factor']).replace(  0, np.nan  )  ),
		activity_capacityFactor=     lambda df:  df['activity'] * df['capacity_factor'],
		activity_per_capacity=          lambda df:  df['activity'] / df['capacity'].replace(  0, np.nan  ),
		activity_capacity=      lambda df:  df['activity'] * df['log1p_capacity'],
		capacity_factor_capacity=       lambda df: df['capacity_factor'] * df['log1p_capacity'],
	## 3. interaction features
		state =  lambda df: df['state'].str.replace( ' ', '' ),
		source_type =  lambda df: df['source_type'].str.replace( '_', '' ),
		inter =  lambda df: df.apply( lambda _df: f"{_df['state']}_{_df['source_type']}", axis= 'columns'   ) ,
	
	)
    ## One-hot encoding for state & interaction-field
    .pipe( api.utils.OHE_func, categorical_col= ['state', 'inter']  )
    
    
	.drop( columns= ['emissions_factor'] ) ## as using this field would leakage the data
)
feature_df.shape
feature_df.head()
 

(8849, 244)

In [66]:
feature_OHE_df = api.utils.OHE_func( feature_df, categorical_col= ['state', 'state_sourceType'] )

In [67]:
feature_OHE_df

,emissions_quantity,capacity,capacity_factor,activity,source_type,area,pop2020,log1p_activity,log1p_capacity,log1p_pop2020,log1p_area,log1Pop_density,activity_per_capita,activity_per_area,capacity_per_capita,capacity_density,potential_output,utilization_ratio,activity_capacityFactor,activity_per_capacity,activity_capacity,capacity_factor_capacity,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_DistrictofColumbia,state_Florida,state_Georgia,state_Idaho,state_Illinois,state_Indiana,state_Iowa,state_Kansas,state_Kentucky,state_Louisiana,state_Maine,state_Maryland,state_Massachusetts,state_Michigan,state_Minnesota,state_Mississippi,state_Missouri,state_Montana,state_Nebraska,state_Nevada,state_NewHampshire,state_NewJersey,state_NewMexico,state_NewYork,state_NorthCarolina,state_NorthDakota,state_Ohio,state_Oklahoma,state_Oregon,state_Pennsylvania,state_RhodeIsland,state_SouthCarolina,state_SouthDakota,state_Tennessee,state_Texas,state_Utah,state_Vermont,state_Virginia,state_Washington,state_WestVirginia,state_Wisconsin,state_Wyoming,state_sourceType_Alabama_coal,state_sourceType_Alabama_gas,state_sourceType_Alabama_oil,state_sourceType_Arizona_coal,state_sourceType_Arizona_gas,state_sourceType_Arizona_oil,state_sourceType_Arkansas_biomass,state_sourceType_Arkansas_coal,state_sourceType_Arkansas_gas,state_sourceType_California_coal,state_sourceType_California_gas,state_sourceType_California_oil,state_sourceType_California_other_fossil,state_sourceType_California_waste,state_sourceType_Colorado_coal,state_sourceType_Colorado_gas,state_sourceType_Colorado_oil,state_sourceType_Connecticut_coal,state_sourceType_Connecticut_gas,state_sourceType_Connecticut_oil,state_sourceType_Connecticut_waste,state_sourceType_Delaware_coal,state_sourceType_Delaware_gas,state_sourceType_Delaware_oil,state_sourceType_DistrictofColumbia_gas,state_sourceType_Florida_coal,state_sourceType_Florida_gas,state_sourceType_Florida_oil,state_sourceType_Florida_other_fossil,state_sourceType_Florida_waste,state_sourceType_Georgia_biomass,state_sourceType_Georgia_coal,state_sourceType_Georgia_gas,state_sourceType_Georgia_oil,state_sourceType_Georgia_other_fossil,state_sourceType_Idaho_gas,state_sourceType_Idaho_other_fossil,state_sourceType_Illinois_coal,state_sourceType_Illinois_gas,state_sourceType_Illinois_oil,state_sourceType_Indiana_coal,state_sourceType_Indiana_gas,state_sourceType_Indiana_oil,state_sourceType_Indiana_other_fossil,state_sourceType_Iowa_coal,state_sourceType_Iowa_gas,state_sourceType_Iowa_oil,state_sourceType_Iowa_other_fossil,state_sourceType_Kansas_coal,state_sourceType_Kansas_gas,state_sourceType_Kansas_oil,state_sourceType_Kentucky_coal,state_sourceType_Kentucky_gas,state_sourceType_Kentucky_oil,state_sourceType_Louisiana_biomass,state_sourceType_Louisiana_coal,state_sourceType_Louisiana_gas,state_sourceType_Louisiana_other_fossil,state_sourceType_Louisiana_waste,state_sourceType_Maine_biomass,state_sourceType_Maine_gas,state_sourceType_Maine_oil,state_sourceType_Maine_waste,state_sourceType_Maryland_coal,state_sourceType_Maryland_gas,state_sourceType_Maryland_oil,state_sourceType_Maryland_waste,state_sourceType_Massachusetts_gas,state_sourceType_Massachusetts_oil,state_sourceType_Massachusetts_waste,state_sourceType_Michigan_coal,state_sourceType_Michigan_gas,state_sourceType_Michigan_oil,state_sourceType_Michigan_other_fossil,state_sourceType_Michigan_waste,state_sourceType_Minnesota_coal,state_sourceType_Minnesota_gas,state_sourceType_Minnesota_oil,state_sourceType_Minnesota_waste,state_sourceType_Mississippi_coal,state_sourceType_Mississippi_gas,state_sourceType_Mississippi_oil,state_sourceType_Missouri_coal,state_sourceType_Missouri_gas,state_sourceType_Missouri_oil,state_sourceType_Montana_coal,state_sourceType_Montana_gas,state_sourceType_Montana_other_fossil,state_sourceType_Nebraska_coal,state_sourceType_Nebraska_gas,state_sourceType_Nebraska_oil,state_sourceType_Nevada_coal,state_

In [ ]:
## import local lib
 api.utils


array(['state_Arizona', 'state_Arkansas', 'state_California',
       'state_Colorado', 'state_Connecticut', 'state_Delaware',
       'state_DistrictofColumbia', 'state_Florida', 'state_Georgia',
       'state_Idaho', 'state_Illinois', 'state_Indiana', 'state_Iowa',
       'state_Kansas', 'state_Kentucky', 'state_Louisiana', 'state_Maine',
       'state_Maryland', 'state_Massachusetts', 'state_Michigan',
       'state_Minnesota', 'state_Mississippi', 'state_Missouri',
       'state_Montana', 'state_Nebraska', 'state_Nevada',
       'state_NewHampshire', 'state_NewJersey', 'state_NewMexico',
       'state_NewYork', 'state_NorthCarolina', 'state_NorthDakota',
       'state_Ohio', 'state_Oklahoma', 'state_Oregon',
       'state_Pennsylvania', 'state_RhodeIsland', 'state_SouthCarolina',
       'state_SouthDakota', 'state_Tennessee', 'state_Texas',
       'state_Utah', 'state_Vermont', 'state_Virginia',
       'state_Washington', 'state_WestVirginia', 'state_Wisconsin',
       'state_Wyoming

In [ ]:
# example usage
# df = pd.read_excel('processed_data.xlsx')
# feat_df = make_features_no_time(df, keep_year=False, add_oof_priors=True)
# feat_df.head()
